In [0]:
# Databricks notebook source

from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, IntegerType, DateType, BooleanType, StringType
from pyspark.sql.window import Window

def transform_employee(employee_df):

    windowSpec_rw = Window.partitionBy("BusinessEntityID").orderBy(F.desc("ModifiedDate"))
    employee_df = employee_df.withColumn(
		"rw", F.row_number().over(windowSpec_rw))
    employee_df = employee_df.filter(F.col("rw") == 1).drop("rw")
    employee_df = employee_df.withColumn("processed_timestamp", F.current_timestamp())

    employee_df = employee_df.select(   
      F.col("BusinessEntityID").cast(IntegerType()).alias("BusinessEntityID"),
      F.col("NationalIDNumber").alias("NationalIDNumber"),
      F.col("LoginID").alias("LoginID"),
      F.col("OrganizationNode").alias("OrganizationNode"),
      F.col("OrganizationLevel").alias("OrganizationLevel"),
      F.col("JobTitle").alias("JobTitle"),
      F.col("BirthDate").cast(DateType()).alias("BirthDate"),
      F.col("MaritalStatus").alias("MaritalStatus"),
      F.col("Gender").alias("Gender"),
      F.col("HireDate").cast(DateType()).alias("HireDate"),
      F.col("SalariedFlag").cast(BooleanType()).alias("SalariedFlag"),
      F.col("VacationHours").alias("VacationHours"),
      F.col("SickLeaveHours").alias("SickLeaveHours"),
      F.col("CurrentFlag").cast(BooleanType()).alias("CurrentFlag"),
      F.col("rowguid").alias("rowguid"),
      F.col("ModifiedDate").cast(DateType()).alias("ModifiedDate"),
      F.col("_rescued_data").alias("_rescued_data"),
      F.col("processed_timestamp").alias("processed_timestamp")
    )
                                 
    return employee_df




if __name__ == "__main__":

    employee_tbl = dbutils.widgets.get("employee")
    employee_df = df = spark.read.table(employee_tbl)
    employee_df_tgt = transform_employee(employee_df)
    employee_slv_tbl = dbutils.widgets.get("employee_tgt")
    employee_df_tgt.write.mode("overwrite").format("delta").partitionBy("ModifiedDate").saveAsTable(employee_slv_tbl)
    